# Resume the VM LinkedIn run — physical batch 256

**Preserve 95 completed evaluations. Resume fold 4 after epoch 1 / update 71, then finish fold 5.**
Original VM source: `bfe0ba3b846cb8bd719d43a586dc4d25b36088cd`.
Physical train/eval batch 256, accumulation 1, float32, official 50-epoch cap and patience 3.
All original data, splits, model/LoRA settings, package versions and VM loader behavior remain unchanged.
This continues the VM v1 experiment; it does not migrate it to the newer Kaggle v2 protocol.

1. Create a **private Kaggle Dataset** from `linkedin_vm256_resume_checkpoint.zip` provided with this
   notebook, and attach it using **Add Input**. ZIP and unpacked inputs are supported.
2. Keep this notebook private. Enable **Internet**, select **GPU T4 ×2** (one GPU also works),
   and enable your existing **GH_TOKEN** secret for this notebook.
   Optional existing **HF_TOKEN** / **TABPFN_TOKEN** credentials are used if present.
3. Use **Save Version → Save & Run All** once. Initial validation must retain all 95 original results
   and find the fold-4 checkpoint. There is **no fresh-run fallback**.

TAR runs in a fresh process, with no earlier baseline model fitted in that process.
On two GPUs, fold 4 TAR and fold 5 baselines start concurrently; fold 5 TAR gets another fresh process.
A single fold is never split across devices. Batch/precision/sequence length never change automatically.
The previous Kaggle attempt ran out of memory: **this notebook does not promise batch 256 will fit**.
There is no short training smoke test; the actual resumed TAR is monitored. An OOM stops and exports saved work.

Live R² tables print after completed folds. Results/checkpoint ZIPs refresh every 15 minutes.
There is no notebook-imposed time cutoff; Kaggle platform/session/quota limits still apply.
Only the best-checkpoint filesystem path is relocated in the working copy; original JSON is preserved.
Completed run JSONs and the experiment manifest must remain byte-identical to the VM originals.


In [ ]:
import os, sys, json, shutil, subprocess, zipfile, runpy, hashlib
from pathlib import Path
SOURCE = 'bfe0ba3b846cb8bd719d43a586dc4d25b36088cd'
RUN_NAME = 'linkedin_vm256_resume'
RESUME_INPUT = ''  # Optional explicit ZIP or unpacked prepared bundle root under /kaggle/input.
WORK = Path('/kaggle/working')
RUN_ROOT = WORK/RUN_NAME
SCRATCH = Path('/kaggle/temp/linkedin_vm256_resume_runtime')
SCRATCH.mkdir(parents=True,exist_ok=True)
print(subprocess.check_output(['nvidia-smi','--query-gpu=index,name,memory.total','--format=csv'],text=True))
GPUS = [int(s) for s in subprocess.check_output(['nvidia-smi','--query-gpu=index','--format=csv,noheader'],text=True).split()]
assert 1<=len(GPUS)<=2,'Select one or two NVIDIA GPUs.'
RUNTIME = SCRATCH/'resume_linkedin_vm256.py'
RUNTIME_CODE = '"""Resume the original VM experiment unchanged; executed OUTSIDE its pinned source checkout."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.util\nimport json\nimport math\nimport os\nfrom pathlib import Path, PurePosixPath\nimport signal\nimport stat\nimport subprocess\nimport sys\nimport time\nimport zipfile\n\nSOURCE = "bfe0ba3b846cb8bd719d43a586dc4d25b36088cd"\nPROTOCOL = "linkedin_vm_physical256_v1"\nRUN_NAME = "linkedin_vm256_resume"\nMODELS = {"tabm": "TabM", "cat": "CatBoost", "light": "LightGBM",\n          "tabpfnv2": "TabPFN-v2", "tabpfnv2p5": "TabPFN-v2p5"}\nCONDITIONS = ["structured", "unstructured", "joint_frozen", "joint_tar"]\nGRID = {(m, c, f) for m in MODELS for c in CONDITIONS for f in range(6)}\nMINIMUM = {(m, c, f) for m, c, f in GRID if f < 4 or (f == 4 and c != "joint_tar")}\nCHECKPOINT_FILES = ["model.safetensors", "optimizer.pt", "scheduler.pt", "rng_state.pth",\n                    "training_args.bin", "trainer_state.json"]\n\n\ndef sha(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef restore_input(source, root):\n    """Require the audited VM-specific bundle. Never overwrite working output or start fresh."""\n    import shutil\n    source, root = Path(source), Path(root).resolve()\n    if root.exists() and any(root.iterdir()):\n        raise ValueError("Refusing to overwrite working output")\n    root.parent.mkdir(parents=True, exist_ok=True)\n    if source.is_dir():\n        if source.is_symlink() or any(p.is_symlink() for p in source.rglob("*")):\n            raise ValueError("Unexpected input symlink")\n        if not (source / "transfer_index.json").is_file():\n            raise ValueError("Select the prepared VM resume bundle root")\n        shutil.copytree(source, root, dirs_exist_ok=True)\n    else:\n        with zipfile.ZipFile(source) as archive:\n            names = set()\n            for info in archive.infolist():\n                p = PurePosixPath(info.filename)\n                if (info.filename in names or p.is_absolute() or ".." in p.parts or "\\\\" in info.filename\n                        or stat.S_ISLNK(info.external_attr >> 16)\n                        or not (root.parent / info.filename).resolve().is_relative_to(root)):\n                    raise ValueError("Unsafe or wrong experiment ZIP member: " + info.filename)\n                names.add(info.filename)\n            if root.name + "/transfer_index.json" not in names:\n                raise ValueError("This is not the prepared VM checkpoint bundle")\n            archive.extractall(root.parent)\n    index = json.loads((root / "transfer_index.json").read_text())\n    verified = root / "transfer_metadata/input_verified.json"\n    if verified.exists():\n        if json.loads(verified.read_text()).get("index_sha256") != sha(root / "transfer_index.json"):\n            raise ValueError("Input verification record/index differs")\n    else:\n        for name, digest in index["initial_checkpoint_sha256"].items():\n            path = root / name\n            if not path.resolve().is_relative_to(root) or sha(path) != digest:\n                raise ValueError("Transferred checkpoint file differs: " + name)\n        write_json(verified, dict(index_sha256=sha(root / "transfer_index.json"), checkpoint_integrity_verified=True))\n    validate_resume(root)\n\n\ndef run_path(output, identity):\n    model, condition, fold = identity\n    return output / "folds" / f"fold{fold}" / "runs" / f"{model}_{condition}_10k_fold{fold}.json"\n\n\ndef validate_resume(root):\n    root = Path(root)\n    output = root / "linkedin_salary"\n    index = json.loads((root / "transfer_index.json").read_text())\n    manifest_path = output / "manifest.json"\n    manifest = json.loads(manifest_path.read_text())\n    if (index.get("source_revision") != SOURCE or manifest.get("source_revision") != SOURCE\n            or manifest.get("protocol") != PROTOCOL or manifest.get("dataset") != "linkedin_salary"\n            or (root / "source_revision.txt").read_text().strip() != SOURCE\n            or sha(manifest_path) != index["manifest_sha256"]):\n        raise ValueError("Original VM source/protocol/manifest differs; no migration or fresh fallback")\n    args = manifest["training_arguments"]\n    for key, expected in dict(per_device_train_batch_size=256, per_device_eval_batch_size=256,\n            gradient_accumulation_steps=1, num_train_epochs=50, learning_rate=.0001,\n            weight_decay=.001, fp16=False, bf16=False, restore_callback_states_from_checkpoint=True).items():\n        if args.get(key) != expected:\n            raise ValueError("Original training settings differ: " + key)\n    if len(index["original_run_sha256"]) != 95:\n        raise ValueError("The audited 95-result inventory is required")\n    for name, digest in index["original_run_sha256"].items():\n        path = root / name\n        if not path.resolve().is_relative_to(root.resolve()) or not path.is_file() or sha(path) != digest:\n            raise ValueError("A previously completed evaluation changed or is missing: " + name)\n    actual = set()\n    for path in output.glob("folds/fold*/runs/*.json"):\n        row = json.loads(path.read_text())\n        identity = (row.get("short_name"), row.get("condition"), row.get("fold"))\n        if identity not in GRID or identity in actual or path != run_path(output, identity):\n            raise ValueError("Unexpected/duplicate result identity: " + str(path))\n        actual.add(identity)\n        model, condition, fold = identity\n        expected = dict(status="ok", dataset="linkedin_salary", model=MODELS[model], task="regression",\n            size="10k", protocol=PROTOCOL, git=SOURCE, multabench_commit=manifest["multabench_commit"],\n            n_train=10000, n_test=5000, n_features=12 if condition == "structured" else 2 if condition == "unstructured" else 14,\n            text_encoder=None if condition == "structured" else "e5-small-tar" if condition == "joint_tar" else "e5-small")\n        if any(row.get(k) != v for k, v in expected.items()):\n            raise ValueError("Saved evaluation failed identity/provenance/status validation: " + str(path))\n        if (not all(isinstance(row.get(k), (float, int)) and not isinstance(row[k], bool) and math.isfinite(row[k])\n                    for k in ("test_score", "test_error", "r2")) or row["r2"] > 1 + 1e-8\n                or row["test_error"] < 0 or not math.isclose(row["test_score"], -row["test_error"], abs_tol=1e-8)):\n            raise ValueError("Invalid regression metrics: " + str(path))\n        if condition == "joint_tar" and (row.get("tar_parameters") != manifest["tar_parameters"]\n                                        or row.get("training_arguments") != args):\n            raise ValueError("Saved TAR parameters differ: " + str(path))\n    if not MINIMUM <= actual:\n        raise ValueError("Previously completed VM evaluations are missing")\n    for fold in (4, 5):\n        cache = output / f"folds/fold{fold}/checkpoints"\n        for directory in (cache / "trainer").glob("checkpoint-*"):\n            if directory.is_dir() and any(not (directory / name).is_file() or (directory / name).stat().st_size == 0\n                                         for name in CHECKPOINT_FILES):\n                raise ValueError("Incomplete checkpoint directory; preserve and inspect it: " + str(directory))\n        if (cache / "complete.json").exists():\n            required = ["adapter/adapter_model.safetensors", "adapter/adapter_config.json", "tokenizer/tokenizer_config.json"]\n            if any(not (cache / name).is_file() for name in required):\n                raise ValueError("Completed TAR adapter/tokenizer is incomplete")\n    if any((m, "joint_tar", 4) not in actual for m in MODELS):\n        cache = output / "folds/fold4/checkpoints"\n        if not (cache / "complete.json").exists():\n            checkpoints = sorted((cache / "trainer").glob("checkpoint-*/trainer_state.json"),\n                                 key=lambda p: int(p.parent.name.split("-")[-1]))\n            if not checkpoints:\n                raise ValueError("Fold 4 checkpoint required; refusing to restart its TAR from scratch")\n            latest = checkpoints[-1]\n            state = json.loads(latest.read_text())\n            if (state.get("global_step", 0) < 71 or not 1 <= state.get("epoch", 0) <= 50\n                    or latest.parent.name != "checkpoint-" + str(state.get("global_step"))):\n                raise ValueError("Fold 4 checkpoint predates the transferred step 71")\n            for name in CHECKPOINT_FILES:\n                p = latest.parent / name\n                if not p.is_file() or p.stat().st_size == 0:\n                    raise ValueError("Incomplete fold 4 checkpoint: " + str(p))\n    return dict(saved_evaluations=len(actual), remaining=120-len(actual), original_results_preserved=95,\n                source_revision=SOURCE, physical_batch=256, accumulation=1)\n\n\ndef relocate_checkpoint_paths(root):\n    """Rewrite only best-checkpoint LOCATION in the working copy; keep all training state intact."""\n    root = Path(root).resolve()\n    changed = []\n    for state_path in root.glob("linkedin_salary/folds/fold*/checkpoints/trainer/checkpoint-*/trainer_state.json"):\n        state = json.loads(state_path.read_text())\n        best = state.get("best_model_checkpoint")\n        if not best:\n            continue\n        name = PurePosixPath(best.replace("\\\\", "/")).name\n        if not name.startswith("checkpoint-") or not name[len("checkpoint-"):].isdigit():\n            raise ValueError("Unexpected best checkpoint path")\n        destination = state_path.parent.parent / name\n        if not (destination / "model.safetensors").is_file():\n            raise ValueError("The saved best checkpoint is missing: " + best)\n        if str(destination) == best:\n            continue\n        backup = root / "transfer_metadata/original_trainer_states" / state_path.relative_to(root)\n        if not backup.exists():\n            backup.parent.mkdir(parents=True, exist_ok=True)\n            backup.write_bytes(state_path.read_bytes())\n        state["best_model_checkpoint"] = str(destination)\n        write_json(state_path, state)\n        changed.append(str(state_path.relative_to(root)))\n    return changed\n\n\ndef original_runtime(project):\n    project = Path(project).resolve()\n    sys.path.insert(0, str(project / "scripts"))\n    spec = importlib.util.spec_from_file_location("original_vm256", project / "scripts/run_full_linkedin_vm.py")\n    vm = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(vm)\n    from tt_curation import REPO_ROOT\n    if REPO_ROOT.resolve() != project:\n        raise ValueError("Editable installation points to another project")\n    if subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=project, text=True).strip() != SOURCE:\n        raise ValueError("Original immutable Git checkout required")\n    return vm\n\n\ndef verify_environment(root, project):\n    validate_resume(root)\n    vm = original_runtime(project)\n    data, config = vm.verified_data()\n    # Includes exact original package versions, all 13 code hashes, data and six splits.\n    vm.initialize_manifest(Path(root) / "linkedin_salary", vm.experiment_settings(data, config))\n    return vm, data\n\n\ndef worker(root, project, fold, phase):\n    import torch\n    vm, data = verify_environment(root, project)\n    vm.require_gpu()\n    output = Path(root) / "linkedin_salary"\n    destination = output / "folds" / f"fold{fold}"\n    from tt_curation.conditions import Condition\n    from tt_curation.runner import run_all\n    selected = [Condition.JOINT_TAR] if phase == "tar" else list(Condition)[:3]\n    expected = [(model, str(condition), fold) for model in MODELS for condition in selected]\n    if all(run_path(output, identity).exists() for identity in expected):\n        print(f"Fold {fold} {phase}: already complete; preserved.", flush=True)\n        return\n    started = time.monotonic()\n    torch.cuda.reset_peak_memory_stats()\n    succeeded = False\n    with vm.GPUMonitor(destination / f"gpu_memory_{phase}_kaggle.csv") as monitor:\n        try:\n            with vm.guarded_tar(destination / "checkpoints"), vm.fail_fast(destination):\n                # Fresh process for every phase. No baseline estimator or probe is fitted in the TAR process.\n                run_all(data, destination, models=list(MODELS), conditions=selected, folds=[fold],\n                        size="10k", device="cuda", frozen_encoder="e5-small")\n            validate_resume(root)\n            if not all(run_path(output, identity).exists() for identity in expected):\n                raise RuntimeError("Phase did not produce all required evaluations")\n            succeeded = True\n        finally:\n            record = dict(phase=phase, seconds=time.monotonic()-started, completed=succeeded,\n                peak_torch_allocated_mib=torch.cuda.max_memory_allocated()/1024**2,\n                sampled_device_peak_mib=monitor.peak, sampling_error=monitor.error,\n                note="This Kaggle phase attempt; may include reused runs/checkpoints.")\n            timing_path = destination / "timing.json"\n            records = json.loads(timing_path.read_text()) if timing_path.exists() else []\n            write_json(timing_path, records + [record])\n\n\ndef export_archives(root, *, live=False):\n    root = Path(root).resolve()\n    paths = sorted(p for p in root.rglob("*") if p.is_file() and not p.is_symlink())\n    if live:\n        ready = set()\n        for p in root.glob("**/trainer/checkpoint-*/trainer_state.json"):\n            try:\n                if p.parent.name == "checkpoint-" + str(json.loads(p.read_text())["global_step"]):\n                    ready.add(p.parent)\n            except (OSError, ValueError, KeyError):\n                pass\n        paths = [p for p in paths if p.suffix not in (".tmp", ".log") and\n                 all(q in ready for q in p.parents if q.name.startswith("checkpoint-") and q.parent.name == "trainer")]\n    for suffix, include_weights in [("results", False), ("checkpoint", True)]:\n        selected = [p for p in paths if include_weights or "checkpoints" not in p.relative_to(root).parts]\n        destination = root.parent / f"{RUN_NAME}_{suffix}.zip"\n        temporary = destination.with_suffix(".zip.tmp")\n        try:\n            before = {p: (p.stat().st_size, p.stat().st_mtime_ns) for p in selected} if live else {}\n            with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as archive:\n                for p in selected:\n                    archive.write(p, (Path(RUN_NAME) / p.relative_to(root)).as_posix())\n            if live and any((p.stat().st_size, p.stat().st_mtime_ns) != before[p] for p in selected):\n                raise OSError("Checkpoint changed during autosave")\n        except OSError:\n            temporary.unlink(missing_ok=True)\n            if not live:\n                raise\n            print("Snapshot changed; retained the previous ZIP.", flush=True)\n            return False\n        temporary.replace(destination)\n        print(f"Saved {destination.name} ({destination.stat().st_size/2**20:.1f} MiB)", flush=True)\n    return True\n\n\ndef report_completed(vm, output, announced):\n    for fold in range(6):\n        if fold not in announced and not vm.audit_runs(output, [fold]):\n            vm.write_reports(output, just_finished=fold)\n            announced.add(fold)\n\n\ndef stop_workers(processes):\n    for p in processes:\n        if p.poll() is None:\n            try: os.killpg(p.pid, signal.SIGINT)\n            except ProcessLookupError: pass\n    deadline = time.monotonic()+40\n    while any(p.poll() is None for p in processes) and time.monotonic() < deadline:\n        time.sleep(1)\n    for p in processes:\n        if p.poll() is None:\n            try: os.killpg(p.pid, signal.SIGKILL)\n            except ProcessLookupError: pass\n        p.wait()\n\n\ndef supervise(root, project, gpus=(0, 1)):\n    vm, _ = verify_environment(root, project)\n    root = Path(root).resolve()\n    output = root / "linkedin_salary"\n    vm.write_reports(output)\n    announced = {f for f in range(6) if not vm.audit_runs(output, [f])}\n    if len(announced) == 6:\n        export_archives(root)\n        return\n    active, processes, handles = {}, [], []\n    jobs = []\n    for fold in (4, 5):\n        if fold in announced: continue\n        baselines = [(m, c, fold) for m in MODELS for c in CONDITIONS[:3]]\n        jobs.append((fold, "tar" if all(run_path(output, i).exists() for i in baselines) else "baselines"))\n    logs = root / "logs"\n    logs.mkdir(exist_ok=True)\n    started = time.monotonic()\n    last_update, last_export = -60, started\n    complete = False\n    failed = None\n    try:\n        while jobs or active:\n            for gpu, (p, handle, fold, phase) in list(active.items()):\n                code = p.poll()\n                if code is None: continue\n                handle.close()\n                del active[gpu]\n                if code:\n                    failed = f"GPU {gpu}, fold {fold}, {phase}: exit {code}. No settings were changed."\n                    break\n                if phase == "baselines": jobs.append((fold, "tar"))\n            if failed: break\n            report_completed(vm, output, announced)\n            for gpu in gpus:\n                if gpu in active or not jobs: continue\n                fold, phase = jobs.pop(0)\n                env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"}\n                handle = (logs / f"fold{fold}_{phase}_gpu{gpu}.log").open("a", encoding="utf-8")\n                handles.append(handle)\n                command = [sys.executable, "-u", str(Path(__file__).resolve()), "--mode", "worker",\n                           "--root", str(root), "--project", str(project), "--fold", str(fold), "--phase", phase]\n                print(f"GPU {gpu}: fold {fold} {phase}; new process, physical batch 256.", flush=True)\n                p = subprocess.Popen(command, env=env, cwd=project, stdout=handle, stderr=subprocess.STDOUT,\n                                     start_new_session=True)\n                processes.append(p)\n                active[gpu] = (p, handle, fold, phase)\n            elapsed = time.monotonic()-started\n            if elapsed-last_update >= 60:\n                count = len(list(output.glob("folds/fold*/runs/*.json")))\n                print(f"{count}/120 saved; active GPU:(fold,phase) "\n                      f"{ {g:(v[2],v[3]) for g,v in active.items()} }; elapsed {elapsed/3600:.2f} h", flush=True)\n                hardware = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",\n                    "--format=csv,noheader"], capture_output=True, text=True, timeout=10)\n                print(hardware.stdout.strip(), flush=True)\n                # Show real TAR progress instead of claiming a short smoke test establishes memory fit.\n                for path in sorted(logs.glob("*tar*.log")):\n                    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()\n                    if lines: print(path.name + ": " + lines[-1], flush=True)\n                last_update = elapsed\n            if time.monotonic()-last_export >= 900:\n                saved = export_archives(root, live=True)\n                last_export = time.monotonic() if saved else time.monotonic()-840\n            if jobs or active: time.sleep(5)\n    except KeyboardInterrupt:\n        failed = "Interrupted; saved work retained."\n    finally:\n        stop_workers(processes)\n        for h in handles: h.close()\n        if failed: print(failed, flush=True)\n        try:\n            validate_resume(root)\n            report_completed(vm, output, announced)\n            complete = vm.write_reports(output)\n        finally:\n            export_archives(root)\n            for path in sorted(logs.glob("*.log")):\n                print("\\n--- " + path.name + " ---\\n" + "\\n".join(path.read_text(encoding="utf-8", errors="replace").splitlines()[-45:]), flush=True)\n    if not complete:\n        raise SystemExit("INCOMPLETE: saved work retained. Inspect logs before resuming; no automatic memory fallback.")\n    print("COMPLETE: all 120 original-VM evaluations audited.", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--mode", choices=["validate", "prepare", "supervise", "worker", "export"], required=True)\n    parser.add_argument("--root", required=True)\n    parser.add_argument("--project")\n    parser.add_argument("--fold", type=int, choices=[4, 5])\n    parser.add_argument("--phase", choices=["baselines", "tar"])\n    parser.add_argument("--gpus", default="0,1")\n    args = parser.parse_args()\n    if args.mode == "export":\n        if Path(args.root).name != RUN_NAME or not (Path(args.root) / "transfer_index.json").is_file():\n            parser.error("Expected the experiment-specific resume root")\n        export_archives(args.root)\n        return\n    print("RESUME CHECK: " + json.dumps(validate_resume(args.root)), flush=True)\n    if args.mode == "validate": return\n    if not args.project: parser.error("--project is required")\n    if args.mode == "prepare":\n        verify_environment(args.root, args.project)\n        changed = relocate_checkpoint_paths(args.root)\n        print("Only relocated checkpoint path fields in working copies:", changed, flush=True)\n    elif args.mode == "supervise":\n        try: gpus = tuple(int(x) for x in args.gpus.split(","))\n        except ValueError: parser.error("Invalid GPU indices")\n        if not gpus or len(set(gpus)) != len(gpus) or any(g < 0 for g in gpus): parser.error("Invalid GPU indices")\n        supervise(args.root, args.project, gpus)\n    else:\n        if args.fold is None or args.phase is None: parser.error("worker requires --fold and --phase")\n        worker(args.root, args.project, args.fold, args.phase)\n\n\nif __name__ == "__main__": main()\n'
RUNTIME.write_text(RUNTIME_CODE,encoding='utf-8')
api = runpy.run_path(str(RUNTIME),run_name='resume_bootstrap')
if (RUN_ROOT/'transfer_index.json').is_file():
    print('Validating existing working output; attached input will not overwrite it.')
else:
    choices=list(Path('/kaggle/input').rglob(RUN_NAME+'_checkpoint.zip'))
    unpacked=[p.parent for p in Path('/kaggle/input').rglob('transfer_index.json')
              if (p.parent/'linkedin_salary/manifest.json').is_file()]
    # Some uploads expose both the ZIP and its adjacent extracted folder.
    choices += [p for p in unpacked if not any(p.resolve()==(z.parent/RUN_NAME).resolve() for z in choices)]
    choices=sorted(set(choices))
    selected=Path(RESUME_INPUT) if RESUME_INPUT else choices[0] if len(choices)==1 else None
    assert selected is not None,'Attach exactly one prepared VM resume input, or set RESUME_INPUT: '+str(choices)
    api['restore_input'](selected,RUN_ROOT)
print('VM RESUME VERIFIED:',json.dumps(api['validate_resume'](RUN_ROOT),indent=2),flush=True)
provenance=RUN_ROOT/'transfer_metadata/continuation_provenance.json'
record=dict(original_source=SOURCE,runtime_sha256=hashlib.sha256(RUNTIME_CODE.encode()).hexdigest(),
            physical_batch=256,accumulation=1,tar_uses_fresh_process=True)
if provenance.exists():
    assert json.loads(provenance.read_text())==record,'Continuation harness differs; preserve output and investigate.'
else: api['write_json'](provenance,record)


In [ ]:
import base64
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
try: gh_token=secrets.get_secret('GH_TOKEN')
except Exception: raise RuntimeError('Enable GH_TOKEN for THIS notebook in Add-ons > Secrets.') from None
PROJECT=SCRATCH/'original_source'
ENV={**os.environ,'MPLBACKEND':'Agg','HF_HOME':str(SCRATCH/'huggingface'),
     'TORCH_HOME':str(SCRATCH/'torch'),'UV_CACHE_DIR':str(SCRATCH/'uv_cache'),
     'XDG_CACHE_HOME':str(SCRATCH/'cache'),'TOKENIZERS_PARALLELISM':'false',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True','PYTHONUNBUFFERED':'1',
     'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2'}
for label in ('HF_TOKEN','TABPFN_TOKEN'):
    try: ENV[label]=secrets.get_secret(label)
    except Exception: print('No optional',label,'; access check will verify availability.')
def run_checked(command,*,env=None,cwd=None):
    p=subprocess.Popen([str(x) for x in command],cwd=cwd or PROJECT,env=env or ENV,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end='',flush=True)
        code=p.wait()
    except KeyboardInterrupt:
        import signal
        p.send_signal(signal.SIGINT); p.wait(); raise
    if code: raise RuntimeError('Command failed; inspect output above. Exit code '+str(code))
try:
    head=subprocess.run(['git','rev-parse','--verify','HEAD'],cwd=PROJECT,capture_output=True,text=True) if (PROJECT/'.git').exists() else None
    if head is not None and head.returncode==0:
        assert head.stdout.strip()==SOURCE,'Existing source checkout differs; preserve it and investigate.'
    else:
        PROJECT.mkdir(parents=True,exist_ok=True)
        if not (PROJECT/'.git').exists(): run_checked(['git','init',PROJECT],cwd=SCRATCH)
        # Credentials are passed only in the fetch process environment, never URLs, arguments or Git config.
        auth=base64.b64encode(('x-access-token:'+gh_token).encode()).decode()
        git_env={**ENV,'GIT_TERMINAL_PROMPT':'0','GIT_TRACE':'0','GIT_CURL_VERBOSE':'0','GIT_TRACE_CURL':'0',
            'GIT_CONFIG_COUNT':'2','GIT_CONFIG_KEY_0':'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0':'AUTHORIZATION: basic '+auth,
            'GIT_CONFIG_KEY_1':'credential.helper','GIT_CONFIG_VALUE_1':''}
        try: run_checked(['git','fetch','--depth','1','--no-tags','https://github.com/diyarh26/text-tabular-curation.git',SOURCE],env=git_env)
        finally: del git_env,auth
        run_checked(['git','checkout','--detach','FETCH_HEAD'])
finally: del gh_token
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT,text=True).strip()==SOURCE
print('Original VM Git source verified:',SOURCE)


In [ ]:
uv=shutil.which('uv')
if not uv:
    run_checked([sys.executable,'-m','pip','install','--quiet','uv'],cwd=SCRATCH)
    uv=shutil.which('uv')
assert uv
PYTHON=SCRATCH/'venv/bin/python'
if not PYTHON.exists(): run_checked([uv,'venv','--python','3.11',PYTHON.parent.parent])
run_checked([uv,'pip','install','--python',PYTHON,'-c',PROJECT/'requirements/constraints.txt',
             '-e',str(PROJECT)+'[bench,dev]','autogluon-features==1.5.0'])
run_checked([uv,'pip','check','--python',PYTHON])
run_checked([PYTHON,'-u',RUNTIME,'--mode','prepare','--root',RUN_ROOT,'--project',PROJECT],
             env={**ENV,'CUDA_VISIBLE_DEVICES':str(GPUS[0])})
for gpu in GPUS:
    run_checked([PYTHON,'-c',
        "import torch; assert torch.cuda.is_available() and torch.cuda.device_count()==1; "
        "assert torch.ones(4,device='cuda').sum().item()==4; print(torch.__version__,torch.cuda.get_device_name(0)); "
        "from tt_curation.runner import _model_class; [_model_class(m) for m in ['tabm','cat','light','tabpfnv2','tabpfnv2p5']]; print('Five learner imports verified')"],
        env={**ENV,'CUDA_VISIBLE_DEVICES':str(gpu)})
# Model-access probe runs in a separate process and exits before any TAR worker starts.
probe='\n'.join([
    'import gc,numpy as np,torch',
    'from tabpfn import TabPFNRegressor',
    'from tabpfn.constants import ModelVersion',
    'X=np.random.default_rng(42).normal(size=(32,4)); y=X[:,0]',
    "for v in ['v2','v2.5']:",
    "    model=TabPFNRegressor.create_default_for_version(version=ModelVersion(v),device='cuda',random_state=42,ignore_pretraining_limits=True)",
    '    model.fit(X,y); assert np.isfinite(model.predict(X[:2])).all()',
    "    print('Regression checkpoint access:',v,flush=True)",
    '    del model; gc.collect(); torch.cuda.empty_cache()',
])
run_checked([PYTHON,'-u','-c',probe],env={**ENV,'CUDA_VISIBLE_DEVICES':str(GPUS[0])})
print('Original manifest matched. Actual production TAR will test memory; no short smoke-test claim.',flush=True)


In [ ]:
from IPython.display import display,FileLink
print('Preserving all 95 originals. Resume fold 4 from checkpoint; run only missing fold-5 work.',flush=True)
print('Physical batch 256; accumulation 1; fresh process for TAR. No automatic fallback or time cutoff.',flush=True)
try:
    run_checked([PYTHON,'-u',RUNTIME,'--mode','supervise','--root',RUN_ROOT,'--project',PROJECT,
                 '--gpus',','.join(map(str,GPUS))],env={**ENV,'CUDA_VISIBLE_DEVICES':str(GPUS[0])})
finally:
    try: run_checked([PYTHON,'-u',RUNTIME,'--mode','export','--root',RUN_ROOT])
    finally:
        for suffix in ('results','checkpoint'):
            p=WORK/(RUN_NAME+'_'+suffix+'.zip')
            if p.exists(): display(FileLink(str(p)))
print((RUN_ROOT/'linkedin_salary/summary.txt').read_text())
